# How your lakehouse design decisions affect Direct Lake performance - live demos

FabCon Europe 2026, Barcelona. Mosaic Events Group ticket sales, 500 million rows per table.

**One baseline, one variable per variant.** Every table below holds the same 500M rows with the same schema. Each variant changes exactly one thing, so every comparison has exactly one explanation.

| Variant | What is different | Files | Row groups |
|---|---|---|---|
| `base` | nothing - compacted, V-Order on, unpartitioned | 8 | 16 |
| `many_files` | file count only (healthy 1M-row row groups) | 500 | 500 |
| `small_rowgroups` | row group size only (compared with `novorder`) | 8 | ~3,700 |
| `fragmented` | everything wrong: tiny files, tiny row groups, no V-Order | 4,000 | 4,000 |
| `novorder` | V-Order off, otherwise identical to `base` | 8 | 16 |
| `part_month` | partitioned by month_key (60 values) | 60 | 60 |
| `part_date` | partitioned by date_key (1,826 values) | 1,826 | 1,826 |

**How every timing is taken.** The model is put into a true cold state (`clearValues` then `full` refresh, about 30 seconds), the query runs, the wall-clock time is recorded. After the reset the two dimension tables are loaded once, so no fact query carries their cost. Every query is wrapped in a filter that changes on each call, so the formula engine's result cache can never answer it - what you see is VertiPaq work on the fact table. The one exception is labelled in Demo 6, where the result cache is the point.

**Run order on stage:** Setup once (before the session), then any demo in any order. Every demo that writes records its table's version immediately before the write. The restore sits in a separate cleanup cell right after the demo, so you can run it while you talk over the result; always run it before the next demo.


## The three reference queries

The instrument matters as much as the table. These three queries were calibrated in rehearsal on the `base` table.

**Light** - three numeric columns through one relationship. Cold about 1 to 4 seconds, warm 0.6 seconds. This is the instrument for every layout comparison: it is cheap enough that transcoding dominates, so differences between tables are differences in layout.

```dax
EVALUATE
SUMMARIZECOLUMNS (
    dim_event[event_category],
    "revenue", SUM ( fact[gross_amount] ),
    "tickets", SUM ( fact[quantity] )
)
```

**Medium** - three string columns and four measures, no distinct count. Seven columns to transcode instead of three. Used where the point is which columns become resident, not how fast one table is.

```dax
EVALUATE
SUMMARIZECOLUMNS (
    fact[venue_city],
    fact[ticket_type],
    fact[payment_method],
    "revenue",   SUM ( fact[gross_amount] ),
    "tickets",   SUM ( fact[quantity] ),
    "avg_price", AVERAGE ( fact[unit_price] ),
    "orders",    COUNTROWS ( fact )
)
```

**Heavy** - the medium query plus a distinct count over `customer_key` (2 million distinct values) and a date attribute. Shown here, never run live: in rehearsal it took 150 to 230 seconds cold on every variant and about 150 seconds warm. The distinct count over a high-cardinality column costs more than any layout mistake in this session. That is a finding, not an instrument.

```dax
EVALUATE
SUMMARIZECOLUMNS (
    fact[venue_city],
    fact[ticket_type],
    fact[payment_method],
    dim_date[year],
    "revenue",   SUM ( fact[gross_amount] ),
    "tickets",   SUM ( fact[quantity] ),
    "avg_price", AVERAGE ( fact[unit_price] ),
    "customers", DISTINCTCOUNT ( fact[customer_key] )
)
```

**Cache bypass** - every timed query is wrapped like this, with `n` incremented per call:

```dax
EVALUATE CALCULATETABLE ( <query>, dim_event[event_key] <> -n )
```


---
## Setup - before the session

Install, configuration, helpers, pristine snapshot, Delta summaries. The model creation cell is only needed the first time.

The Delta summaries are cached in the lakehouse (`Files/demo_cache/delta_summaries.json`). Building the cache takes about 20 minutes because Delta Analyzer opens every file of `fragmented` and `part_date`; it is done once. After that, every session - including the one on stage - loads it in a second. The cache stays valid as long as the tables are not regenerated with notebook 01, and the load cell checks that on every run.

**Conference-day setup, about two minutes:** install, config, helpers, pristine snapshot, Delta summaries (loads from cache), increment generator.


In [ ]:
%pip install semantic-link-labs --quiet

In [3]:
import time, statistics
import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake
from pyspark.sql import functions as F

DATASET   = "mosaic_direct_lake_demo"
WORKSPACE = None                       # None = current workspace
LAKEHOUSE = "LH_FabConBarcelona"
SCHEMA    = "dbo"
PREFIX    = "fact_ticket_sales"

VARIANTS = ["base", "many_files", "small_rowgroups", "fragmented", "novorder", "part_month", "part_date"]
DIMS = ["dim_date", "dim_event"]
def T(variant): return f"{PREFIX}_{variant}"
FACTS = [T(v) for v in VARIANTS]
ALL_TABLES = DIMS + FACTS
BASE = T("base")

# Slow 500M-row writes. False = show the rehearsal evidence rows in the same table instead of running the write.
LIVE_SLOW_WRITES = False                      # Demo 5: MERGE and OVERWRITE, about 4 minutes each
DELETE_DATE_KEY = 20210101                    # Demo 4 Part B: oldest day in the dataset (the generator is deterministic), so no scan to find it
LADDER_TABLE = "fragmented"                   # Demo 6 Part B: a table whose transcoding is expensive enough for the four states to separate in time

# Delta summaries cache. True = rebuild (about 20 minutes) and overwrite the cache - only after regenerating tables with notebook 01.
DELTA_CACHE = "/lakehouse/default/Files/demo_cache/delta_summaries.json"
REBUILD_DELTA_CACHE = False
print(len(ALL_TABLES), "tables in scope; slow writes live:", LIVE_SLOW_WRITES)


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 10, Finished, Available, Finished, False)

9 tables in scope; slow writes live: False


### Semantic model - first time only (skip if the model exists)

In [ ]:
directlake.generate_direct_lake_semantic_model(
    dataset=DATASET,
    tables={t: f"{SCHEMA}.{t}" for t in ALL_TABLES},   # schema-qualified source, bare model names
    source=LAKEHOUSE, source_type="Lakehouse",
    use_sql_endpoint=False,                            # Direct Lake on OneLake
    workspace=WORKSPACE, refresh=True, overwrite=True,
)
with labs.tom.connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    for fact in FACTS:
        tom.add_relationship(from_table=fact, from_column="date_key", to_table="dim_date", to_column="date_key",
                             from_cardinality="Many", to_cardinality="One")
        tom.add_relationship(from_table=fact, from_column="event_key", to_table="dim_event", to_column="event_key",
                             from_cardinality="Many", to_cardinality="One")
print("model and relationships created")


### Helpers

In [4]:
QUERIES = {
    "light": lambda t: f'''SUMMARIZECOLUMNS(dim_event[event_category],
        "revenue", SUM({t}[gross_amount]), "tickets", SUM({t}[quantity]))''',
    "medium": lambda t: f'''SUMMARIZECOLUMNS({t}[venue_city], {t}[ticket_type], {t}[payment_method],
        "revenue", SUM({t}[gross_amount]), "tickets", SUM({t}[quantity]),
        "avg_price", AVERAGE({t}[unit_price]), "orders", COUNTROWS({t}))''',
    "single": lambda t: f'''ROW("revenue", SUM({t}[gross_amount]))''',
}
_nonce = [0]

def timed_query(table, profile="light", label="", bust=True):
    body = QUERIES[profile](table)
    if bust:
        _nonce[0] += 1
        body = f"CALCULATETABLE({body}, dim_event[event_key] <> -{_nonce[0]})"
    t0 = time.perf_counter()
    fabric.evaluate_dax(dataset=DATASET, dax_string="EVALUATE " + body, workspace=WORKSPACE)
    secs = round(time.perf_counter() - t0, 2)
    print(f"  {label or table:34s} {profile:7s} {secs:7.2f}s")
    return secs

def _segments(table):
    df = fabric.evaluate_dax(dataset=DATASET, dax_string="EVALUATE INFO.STORAGETABLECOLUMNSEGMENTS()", workspace=WORKSPACE)
    df.columns = [c.strip("[]") for c in df.columns]
    df = df[df["DIMENSION_NAME"].astype(str) == table]
    df = df[~df["COLUMN_ID"].astype(str).str.startswith("RowNumber")]
    return df.assign(resident=(pd.to_numeric(df["ISRESIDENT"], errors="coerce").fillna(0) == 1).astype(int),
                     temp=pd.to_numeric(df["TEMPERATURE"], errors="coerce"))

def residency(table):
    df = _segments(table)
    res = df[df["resident"] == 1]
    return dict(segments=len(df), resident=len(res),
                resident_pct=round(100 * len(res) / max(len(df), 1)),
                columns_resident=int(res["COLUMN_ID"].nunique()),
                avg_temperature=round(float(res["temp"].mean()), 2) if len(res) else 0.0)

def residency_by_column(table):
    df = _segments(table)
    g = (df.groupby("COLUMN_ID").agg(segments=("SEGMENT_NUMBER", "count"), resident=("resident", "sum"),
                                     temperature=("temp", "max")).reset_index())
    g["column"] = g["COLUMN_ID"].str.replace(r" \(\d+\)$", "", regex=True)
    g["temperature"] = g["temperature"].round(2)
    display(g[["column", "segments", "resident", "temperature"]].sort_values(["resident", "column"], ascending=[False, True]))

def reframe():
    labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")

def warm_dims():
    # After clearValues the dimensions are cold too. Load them (and the relationship index) before timing any fact,
    # so the first fact query in a race does not carry a cost the others skip. Fact tables stay cold.
    fabric.evaluate_dax(dataset=DATASET, workspace=WORKSPACE,
                        dax_string="EVALUATE SUMMARIZECOLUMNS(dim_event[event_category], dim_date[year])")

def cold_reset():
    labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="clearValues")
    labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")
    warm_dims()
    print("  cold state induced (dimensions warm, facts cold)")

def analyze(table):
    return labs.delta_analyzer(table, lakehouse=LAKEHOUSE, workspace=WORKSPACE, schema=SCHEMA,
                               visualize=False, column_stats=False, _show_progress=False)["Summary"]

def metrics(table):
    row = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").collect()[0]
    m = dict(row["operationMetrics"])
    def pick(*keys):
        for k in keys:
            if k in m: return int(m[k])
        return None
    return dict(operation=row["operation"], version=row["version"],
                files_removed=pick("numRemovedFiles", "numTargetFilesRemoved"),
                files_added=pick("numAddedFiles", "numTargetFilesAdded", "numFiles"),
                rows_deleted=pick("numDeletedRows", "numTargetRowsDeleted"),
                rows_updated=pick("numUpdatedRows", "numTargetRowsUpdated"),
                rows_inserted=pick("numOutputRows", "numTargetRowsInserted"))

_tick = [time.perf_counter()]
def tick_start(): _tick[0] = time.perf_counter()
def tick(label):
    # step timer: prints the seconds since the previous tick, so a rehearsal shows where a cell spends its time
    now = time.perf_counter(); print(f"  [{now - _tick[0]:6.1f}s] {label}"); _tick[0] = now

PRISTINE = {}
def snapshot(table): PRISTINE[table] = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").collect()[0]["version"]
def restore(table):
    if table not in PRISTINE:
        raise RuntimeError(f"No version recorded for {table}. Every demo calls snapshot() right before it writes; "
                           f"for a manual fix use DESCRIBE HISTORY {table} and RESTORE TABLE ... TO VERSION AS OF <version>.")
    spark.sql(f"RESTORE TABLE {table} TO VERSION AS OF {PRISTINE[table]}")
    print(f"  {table} restored to version {PRISTINE[table]}")

def delta_row(variant):
    s = DELTA[variant].iloc[0]
    return dict(files=int(s["Parquet Files"]), row_groups=int(s["Row Groups"]),
                avg_rows_per_row_group=int(s["Avg Rows Per Row Group"]), vorder=bool(s["VOrder Enabled"]))

def results_table(rows, baseline=None):
    df = pd.DataFrame(rows)
    if baseline is not None and "cold_s" in df.columns:
        b = df.loc[df["variant"] == baseline, "cold_s"].iloc[0]
        df["x_slower_cold"] = (df["cold_s"] / b).round(1)
    display(df)
print("helpers ready")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 11, Finished, Available, Finished, False)

helpers ready


In [4]:
# Pristine snapshot of every fact - used by the final Reset cell only.
# Each demo that writes records its table's version right before its own write, so demos do not depend on this cell.
for t in FACTS: snapshot(t)
print(PRISTINE)


StatementMeta(, 60d7673f-80cf-4f9b-8132-7adfdae04a8d, 11, Finished, Available, Finished, False)

{'fact_ticket_sales_base': 72, 'fact_ticket_sales_many_files': 0, 'fact_ticket_sales_small_rowgroups': 0, 'fact_ticket_sales_fragmented': 14, 'fact_ticket_sales_novorder': 0, 'fact_ticket_sales_part_month': 26, 'fact_ticket_sales_part_date': 25}


In [5]:
# Delta summaries: load from the lakehouse cache, or build and save it (about 20 minutes) if it is missing or a rebuild is requested.
import json, os
from IPython.utils import io

def build_delta_cache():
    cache = {}
    for v in VARIANTS:
        with io.capture_output():   # the Delta Analyzer progress widget ignores _show_progress and overflows the output
            cache[v] = analyze(T(v))
        print(f"  {v:16s} analyzed")
    os.makedirs(os.path.dirname(DELTA_CACHE), exist_ok=True)
    with open(DELTA_CACHE, "w") as f:
        json.dump({v: df.to_dict(orient="records") for v, df in cache.items()}, f, default=str)
    print(f"  cache saved to {DELTA_CACHE}")
    return cache

if REBUILD_DELTA_CACHE or not os.path.exists(DELTA_CACHE):
    print("Building the Delta summaries cache - about 20 minutes")
    DELTA = build_delta_cache()
else:
    with open(DELTA_CACHE) as f:
        DELTA = {v: pd.DataFrame(rows) for v, rows in json.load(f).items()}
    missing = [v for v in VARIANTS if v not in DELTA]
    assert not missing, f"Cache has no entry for {missing} - set REBUILD_DELTA_CACHE = True and re-run"
    print(f"Loaded Delta summaries for {len(DELTA)} tables from cache")

# Staleness check (a few seconds): the file count in each table's current version must match the cache.
# A mismatch means the table was regenerated, or a demo was interrupted before it restored its table.
stale = []
for v in VARIANTS:
    now = spark.sql(f"DESCRIBE DETAIL {T(v)}").collect()[0]["numFiles"]
    cached = delta_row(v)["files"]
    if now != cached:
        stale.append(f"{v}: cache {cached} files, table now {now}")
if stale:
    print("WARNING - cache and tables disagree:")
    for line in stale: print("  " + line)
    print("If a demo was interrupted, restore that table (DESCRIBE HISTORY, then RESTORE TABLE). If tables were regenerated, set REBUILD_DELTA_CACHE = True.")
else:
    print("All tables match the cache")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 12, Finished, Available, Finished, False)

All tables match the cache


In [6]:
display(pd.DataFrame([dict(variant=v, **delta_row(v)) for v in VARIANTS]))

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b3b23d2-0818-4d1f-bd47-f28a41d6fddb)

### Increment generator

Builds one new day of sales (about 550K rows, deterministic) for the append demos.


In [7]:
SEED = 20260914
CITIES = ["Barcelona", "Lisbon", "Berlin", "Amsterdam", "Vienna", "Copenhagen",
          "Madrid", "Milan", "Paris", "Prague", "Dublin", "Stockholm"]
TICKET_TYPES, TICKET_PRICES = ["General", "VIP", "Student", "Online", "Workshop"], [299.00, 899.00, 99.00, 149.00, 499.00]
PAYMENT_METHODS = ["Card", "Invoice", "PayPal", "Voucher"]
def _h(c, salt, mod): return F.pmod(F.xxhash64(c, F.lit(salt), F.lit(SEED)), F.lit(mod))

def build_increment(table, n=550_000, date="2026-01-01", date_key=20260101, month_key=202601):
    base_id = spark.sql(f"SELECT MAX(sale_id) AS m FROM {table}").collect()[0]["m"] + 1
    inc = (spark.range(base_id, base_id + n).withColumnRenamed("id", "sale_id")
        .withColumn("sale_date", F.to_date(F.lit(date))).withColumn("date_key", F.lit(date_key)).withColumn("month_key", F.lit(month_key)))
    tt = _h(F.col("sale_id"), 3, 5)
    return (inc.withColumn("event_key", (_h(F.col("sale_id"), 2, 500) + 1).cast("int"))
        .withColumn("venue_city", F.element_at(F.array(*[F.lit(c) for c in CITIES]), (F.pmod(F.col("event_key"), F.lit(12)) + 1).cast("int")))
        .withColumn("ticket_type", F.element_at(F.array(*[F.lit(t) for t in TICKET_TYPES]), (tt + 1).cast("int")))
        .withColumn("payment_method", F.element_at(F.array(*[F.lit(p) for p in PAYMENT_METHODS]), (_h(F.col("sale_id"), 4, 4) + 1).cast("int")))
        .withColumn("customer_key", _h(F.col("sale_id"), 5, 2_000_000).cast("int"))
        .withColumn("quantity", (_h(F.col("sale_id"), 6, 6) + 1).cast("int"))
        .withColumn("unit_price", F.round(F.element_at(F.array(*[F.lit(p) for p in TICKET_PRICES]), (tt + 1).cast("int"))
                    * (F.lit(1.0) + F.pmod(F.col("event_key"), F.lit(20)) / F.lit(100.0)), 2).cast("decimal(10,2)"))
        .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2).cast("decimal(12,2)"))
        .select("sale_id", "sale_date", "date_key", "month_key", "event_key", "venue_city", "ticket_type",
                "payment_method", "customer_key", "quantity", "unit_price", "gross_amount"))
print("increment generator ready")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 14, Finished, Available, Finished, False)

increment generator ready


---
# Demo 1 - number of parquet files

**What we compare.** Three tables, same 500M rows, same V-Order, same healthy row groups where possible. Only the number of files differs: 8, 500, and 4,000.

**What happens.** From cold, the light query runs once on each table. Then it runs again while everything is warm.

**What to expect.** Cold: `base` about 1s, `many_files` about 2s, `fragmented` 7 to 15s. Warm: all three around 0.4s. The entire difference is transcoding - the time to turn parquet into VertiPaq segments - and it scales with the number of files that have to be opened, read and merged. Once the segments are in memory, the layout stops mattering.

**Stage time:** about 1.5 minutes.


In [8]:
demo1 = ["base", "many_files", "fragmented"]
rows = {v: dict(variant=v, **delta_row(v)) for v in demo1}
cold_reset()
for v in demo1: rows[v]["cold_s"] = timed_query(T(v), "light", f"{v} cold")
print("--- warm repeat")
for v in demo1: rows[v]["warm_s"] = timed_query(T(v), "light", f"{v} warm")
results_table(list(rows.values()), baseline="base")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 15, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  base cold                          light      0.68s
  many_files cold                    light      2.30s
  fragmented cold                    light     15.28s
--- warm repeat
  base warm                          light      0.47s
  many_files warm                    light      0.33s
  fragmented warm                    light      0.32s


SynapseWidget(Synapse.DataFrame, 9150f5af-5dbf-4de1-aa49-d4ea53de2c2a)

---
# Demo 2 - row group size

**What we compare.** `novorder` and `small_rowgroups`: the same 8 files, the same data, V-Order off on both. The only difference is how many row groups each file contains - 16 in total versus roughly 3,700. `small_rowgroups` simulates a table written by an external engine (a shortcut, pyarrow, delta-rs): Fabric Spark does not produce this shape, but plenty of tables arriving in a lakehouse do.

**What happens.** Cold light query on both.

**What to expect.** `novorder` about 3 to 6s, `small_rowgroups` about 9 to 11s: two to three times slower on byte-identical files. Every row group is a column segment to transcode and a local dictionary to merge. A related fact worth stating: the 5,500-row-group version of this table exceeded the F256 guardrail for row groups per table and would not load at all.

**Stage time:** about 1 minute.


In [9]:
demo2 = ["novorder", "small_rowgroups"]
rows = {v: dict(variant=v, **delta_row(v)) for v in demo2}
cold_reset()
for v in demo2: rows[v]["cold_s"] = timed_query(T(v), "light", f"{v} cold")
results_table(list(rows.values()), baseline="novorder")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 16, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  novorder cold                      light      2.63s
  small_rowgroups cold               light      9.77s


SynapseWidget(Synapse.DataFrame, 03a8beb4-9538-48a9-98d4-d0e332821a11)

---
# Demo 3 - V-Order

**What we compare.** `base` and `novorder`: identical files, identical row groups, identical data. One table property differs.

**What happens.** Cold light query on both, then the medium query on both (string columns are where V-Order's encoding matters most), then residency by column for both.

**What to expect.** Light: about 1s versus 3s - two and a half to three times slower without V-Order. This was the largest single-variable effect in the matrix apart from `fragmented`. The residency tables show the same columns resident in both models; what differs is how long it took to get them there.

**Stage time:** about 2 minutes plus the medium query (calibrate in rehearsal).


In [10]:
demo3 = ["base", "novorder"]
rows = {v: dict(variant=v, **delta_row(v)) for v in demo3}
cold_reset()
for v in demo3: rows[v]["cold_light_s"] = timed_query(T(v), "light", f"{v} cold")
cold_reset()
for v in demo3: rows[v]["cold_medium_s"] = timed_query(T(v), "medium", f"{v} cold")
for v in demo3:
    r = residency(T(v)); rows[v].update(segments=r["segments"], resident=r["resident"], columns_resident=r["columns_resident"])
results_table(list(rows.values()))


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 17, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  base cold                          light      0.61s
  novorder cold                      light      2.72s
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh

SynapseWidget(Synapse.DataFrame, 4e586c70-d646-4208-b4f0-21bf709d8d3a)

In [11]:
for v in demo3:
    print("=" * 60, v); residency_by_column(T(v))


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 18, Finished, Available, Finished, False)

============================================================ base


SynapseWidget(Synapse.DataFrame, 918a33af-adf8-478b-95d6-f8a4087d3c7e)

============================================================ novorder


SynapseWidget(Synapse.DataFrame, 962d0e06-16f0-4c7d-8267-5a72273cbe9c)

---
# Demo 4 - partitioning

**What we compare.** `base` (unpartitioned), `part_month` (60 partitions), `part_date` (1,826 partitions). Same rows, same V-Order.

**Part A - reading.** Cold light query on all three. Direct Lake does no partition pruning and no file skipping, so partitioning cannot make a query faster by elimination. `part_date` pays for 1,826 small files.

**What to expect.** `base` and `part_month` within noise of each other, about 1 to 1.7s in either order across runs. `part_date` 4.5 to 9s, five to eight times slower. Partitioning by a sensible key neither helps nor hurts reads; partitioning by a high-cardinality key hurts because of the 1,826 small files it produced.

**Part B - writing.** Warm all three, delete a single day (274K rows, 0.05 percent) from each, compare the Delta metrics and the time each delete took, reframe, and look at what stayed resident. Everything in the result table is measured live. This is where partitioning earns its keep - and the month key is the right cardinality for it.

**What to expect.** `base`: 8 files removed, 16 added - every file rewritten, 0 percent of the warm segments retained. `part_month`: 1 removed, 1 added, everything retained except the deleted partition. `part_date`: 1 removed, 0 added (the partition was that day), likewise. One day of data invalidated the entire cache of the unpartitioned table and almost none of the partitioned ones. `delete_s` should show the same story in time: the partitioned deletes touch one file each, the unpartitioned one rewrites all eight (to be calibrated on the next rehearsal). `retained_pct` is measured against what was resident before the delete.

**Stage time:** Part A about 1 minute. Part B about 2.5 minutes (estimate - the step timers in the cell print the real breakdown), plus the cleanup cell, about 1 minute, run while you talk over the table.


In [12]:
# Part A - cold light query on all three tables
demo4 = ["base", "part_month", "part_date"]
rows = {v: dict(variant=v, **delta_row(v)) for v in demo4}
cold_reset()
for v in demo4: rows[v]["cold_s"] = timed_query(T(v), "light", f"{v} cold")
results_table(list(rows.values()), baseline="base")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 19, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  base cold                          light      0.58s
  part_month cold                    light      0.73s
  part_date cold                     light      6.68s


SynapseWidget(Synapse.DataFrame, c5991eab-bfe5-478a-89b1-24f3fb30fa70)

In [13]:
# Part B start - the two partitioned deletes live, one at a time, then the slow base delete in the background.
# Run right after Part A, then talk while base runs.
import threading
if "BG_DELETE" in globals() and BG_DELETE["thread"].is_alive():
    raise RuntimeError("A background delete is still running - wait for it, then run Part B")
tick_start()
demo4b = {}
for v in ["part_month", "part_date"]:
    timed_query(T(v), "medium", f"{v} warm-up")
    before = residency(T(v))["resident"]
    snapshot(T(v))
    t0 = time.perf_counter()
    spark.sql(f"DELETE FROM {T(v)} WHERE date_key = {DELETE_DATE_KEY}")
    delete_s = round(time.perf_counter() - t0, 1)
    m = metrics(T(v))
    demo4b[v] = dict(variant=v, files_before=delta_row(v)["files"], files_removed=m["files_removed"],
                     files_added=m["files_added"], rows_deleted=m["rows_deleted"], delete_s=delete_s, resident_before=before)
    tick(f"{v}: warm-up, residency, DELETE ({delete_s}s)")

timed_query(BASE, "medium", "base warm-up")
BG_DELETE = dict(resident_before=residency(BASE)["resident"], done=False, error=None)
snapshot(BASE)
tick("base warm-up, residency, snapshot")

def _delete_base():
    try:
        t0 = time.perf_counter()
        spark.sql(f"DELETE FROM {BASE} WHERE date_key = {DELETE_DATE_KEY}")
        BG_DELETE["delete_s"] = round(time.perf_counter() - t0, 1)
        BG_DELETE["metrics"] = metrics(BASE)
    except Exception as e:
        BG_DELETE["error"] = e
    finally:
        BG_DELETE["done"] = True

BG_DELETE["started"] = time.perf_counter()
BG_DELETE["thread"] = threading.Thread(target=_delete_base, daemon=True)
BG_DELETE["thread"].start()
print(f"  DELETE FROM {BASE} WHERE date_key = {DELETE_DATE_KEY} - running in the background, about 3 minutes")

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 20, Finished, Available, Finished, False)

  part_month warm-up                 medium     0.86s
  [  48.2s] part_month: warm-up, residency, DELETE (39.6s)
  part_date warm-up                  medium     8.24s
  [  18.1s] part_date: warm-up, residency, DELETE (3.5s)
  base warm-up                       medium     0.83s
  [   6.3s] base warm-up, residency, snapshot
  DELETE FROM fact_ticket_sales_base WHERE date_key = 20210101 - running in the background, about 3 minutes


In [14]:
# Part B - wait for the base delete if needed, reframe, compare all three
if "BG_DELETE" not in globals() or "demo4b" not in globals():
    raise RuntimeError("Run the Part B start cell first")
tick_start()
while BG_DELETE["thread"].is_alive():
    print(f"  waiting for the base delete - {time.perf_counter() - BG_DELETE['started']:.0f}s so far")
    BG_DELETE["thread"].join(timeout=15)
if BG_DELETE["error"]:
    raise BG_DELETE["error"]
m = BG_DELETE["metrics"]
demo4b["base"] = dict(variant="base", files_before=delta_row("base")["files"], files_removed=m["files_removed"],
                      files_added=m["files_added"], rows_deleted=m["rows_deleted"], delete_s=BG_DELETE["delete_s"],
                      resident_before=BG_DELETE["resident_before"])
tick(f"base delete finished ({BG_DELETE['delete_s']}s)")

reframe(); tick("reframe")
for v in demo4:
    after = residency(T(v))["resident"]
    demo4b[v].update(resident_after=after, retained_pct=round(100 * after / max(demo4b[v]["resident_before"], 1)))
tick("residency after reframe")
results_table([demo4b[v] for v in demo4])

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 21, Finished, Available, Finished, False)

  waiting for the base delete - 6s so far
  waiting for the base delete - 21s so far
  waiting for the base delete - 36s so far
  waiting for the base delete - 51s so far
  waiting for the base delete - 66s so far
  waiting for the base delete - 81s so far
  waiting for the base delete - 96s so far
  waiting for the base delete - 111s so far
  waiting for the base delete - 126s so far
  waiting for the base delete - 141s so far
  waiting for the base delete - 156s so far
  waiting for the base delete - 171s so far
  waiting for the base delete - 186s so far
  waiting for the base delete - 201s so far
  waiting for the base delete - 216s so far
  waiting for the base delete - 231s so far
  waiting for the base delete - 246s so far
  waiting for the base delete - 261s so far
  [ 256.7s] base delete finished (262.5s)
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic

SynapseWidget(Synapse.DataFrame, aa66ba42-abcf-499a-86c2-091b51b000df)

In [15]:
# Part B cleanup - run while you talk over the table, and before the next demo. Restores all three tables and reframes.
tick_start()
for v in demo4: restore(T(v))
tick("restore")
reframe(); tick("reframe")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 22, Finished, Available, Finished, False)

  fact_ticket_sales_base restored to version 72
  fact_ticket_sales_part_month restored to version 26
  fact_ticket_sales_part_date restored to version 25
  [  37.4s] restore
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  [  16.7s] reframe


---
# Demo 5 - append vs merge vs overwrite

**What we compare.** Three ways of writing the same table, judged by one question: after the model reframes, how much of the warm cache is still there?

**What happens.** Warm `base` with the medium query and show residency by column. Then, for each pattern: write, show the Delta metrics, reframe, show residency. `RESTORE` between patterns.

- **Append** - one new day, about 550K rows. New files only. Runs live: the write takes seconds.
- **Merge** - update 1 percent of rows. Parquet files are immutable, so every file containing a matched row is rewritten. About 4 minutes on 500M rows - runs live only if `LIVE_SLOW_WRITES` is `True`, otherwise the rehearsal evidence is shown.
- **Overwrite** - the same data written back with `mode("overwrite")`. Logically nothing changed. Same rule as merge.

**What to expect.** Append: 100 percent of the warm segments retained - every pre-existing segment survived the reframe; only the new rows are cold. Note also that 550K appended rows produced 32 new row groups of about 17K rows: appends without an OPTIMIZE cadence are how fragmented tables come to exist. Merge: 0 percent retained - the 1 percent update rewrote every file. Overwrite: 0 percent retained, and Delta records it as `CREATE OR REPLACE TABLE AS SELECT`, which is what a nightly full load literally is. The orchestration pattern is a cache decision. Rehearsal evidence rows show `resident_before` of 128 because that run used an eight-column query; the live append shows 112 with the medium query.

**Stage time:** about 2 minutes with the slow writes off, plus the append cleanup cell (about 1 minute, run while you talk). About 12 minutes with the slow writes on.


In [16]:
cold_reset()
timed_query(BASE, "medium", "warm-up")
r0 = residency(BASE); print(" ", r0); residency_by_column(BASE)
demo5 = {}   # pattern -> result row; each demo 5 cell replaces its own row and shows the table so far

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 23, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  warm-up                            medium     0.64s
  {'segments': 192, 'resident': 112, 'resident_pct': 58, 'columns_resident': 7, 'avg_temperature': 0.97}


SynapseWidget(Synapse.DataFrame, e137c9ab-bffa-4a4b-a926-9df54d14ec6c)

In [17]:
# APPEND - live
tick_start()
snapshot(BASE)
build_increment(BASE).write.format("delta").mode("append").saveAsTable(BASE); tick("append")
m = metrics(BASE); reframe(); tick("reframe")
r = residency(BASE)
demo5["append"] = (dict(pattern="append", delta_operation=m["operation"], files_removed=m["files_removed"] or 0, files_added=m["files_added"],
                  resident_before=r0["resident"], resident_after=r["resident"], segments_after=r["segments"],
                  retained_pct=round(100 * r["resident"] / max(r0["resident"], 1))))
residency_by_column(BASE); tick("residency")
results_table(list(demo5.values()))

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 24, Finished, Available, Finished, False)

  [  10.6s] append
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  [  17.5s] reframe


SynapseWidget(Synapse.DataFrame, f585d60d-fed4-4359-9ff4-96c75151496a)

  [  10.0s] residency


SynapseWidget(Synapse.DataFrame, 3c2537c8-1452-40c3-b001-be4004437b4d)

In [18]:
# APPEND cleanup - run while talking over the residency table, and before the merge cell
tick_start()
restore(BASE); tick("restore")
reframe(); tick("reframe")


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 25, Finished, Available, Finished, False)

  fact_ticket_sales_base restored to version 74
  [   9.1s] restore
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  [  16.8s] reframe


In [19]:
# MERGE - live only if LIVE_SLOW_WRITES
if LIVE_SLOW_WRITES:
    timed_query(BASE, "medium", "re-warm"); r0 = residency(BASE)
    snapshot(BASE)
    spark.sql(f'''MERGE INTO {BASE} t
        USING (SELECT sale_id, ROUND(gross_amount * 1.05, 2) AS new_amount FROM {BASE}
               WHERE PMOD(XXHASH64(sale_id, 9), 100) = 0) s
        ON t.sale_id = s.sale_id WHEN MATCHED THEN UPDATE SET t.gross_amount = s.new_amount''')
    m = metrics(BASE); reframe(); r = residency(BASE)
    demo5["merge"] = (dict(pattern="merge (1% of rows)", delta_operation=m["operation"], files_removed=m["files_removed"] or 0, files_added=m["files_added"],
                      resident_before=r0["resident"], resident_after=r["resident"], segments_after=r["segments"],
                      retained_pct=round(100 * r["resident"] / max(r0["resident"], 1))))
    restore(BASE); reframe()
else:
    print("MERGE not run live (LIVE_SLOW_WRITES = False) - row below is the rehearsal result from 11 Sep 2026")
    demo5["merge"] = dict(pattern="merge (1% of rows) - rehearsal 11 Sep 2026", delta_operation="MERGE", files_removed=8, files_added=33,
                          resident_before=128, resident_after=0, segments_after=396, retained_pct=0)
results_table(list(demo5.values()))

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 26, Finished, Available, Finished, False)

MERGE not run live (LIVE_SLOW_WRITES = False) - row below is the rehearsal result from 11 Sep 2026


SynapseWidget(Synapse.DataFrame, 0896a7a9-08a4-4b29-92ed-b83864a3494e)

In [20]:
# OVERWRITE - live only if LIVE_SLOW_WRITES
if LIVE_SLOW_WRITES:
    timed_query(BASE, "medium", "re-warm"); r0 = residency(BASE)
    snapshot(BASE)
    spark.table(BASE).write.format("delta").mode("overwrite").saveAsTable(BASE)
    m = metrics(BASE); reframe(); r = residency(BASE)
    demo5["overwrite"] = (dict(pattern="overwrite (same data)", delta_operation=m["operation"], files_removed=m["files_removed"] or 0, files_added=m["files_added"],
                      resident_before=r0["resident"], resident_after=r["resident"], segments_after=r["segments"],
                      retained_pct=round(100 * r["resident"] / max(r0["resident"], 1))))
    restore(BASE); reframe()
else:
    print("OVERWRITE not run live (LIVE_SLOW_WRITES = False) - row below is the rehearsal result from 11 Sep 2026")
    demo5["overwrite"] = dict(pattern="overwrite (same data) - rehearsal 11 Sep 2026", delta_operation="CREATE OR REPLACE TABLE AS SELECT",
                              files_removed=8, files_added=17, resident_before=128, resident_after=0, segments_after=192, retained_pct=0)
results_table(list(demo5.values()))

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 27, Finished, Available, Finished, False)

OVERWRITE not run live (LIVE_SLOW_WRITES = False) - row below is the rehearsal result from 11 Sep 2026


SynapseWidget(Synapse.DataFrame, 2eb0654a-4f02-4de2-9693-e51f26579ba8)

---
# Demo 6 - memory residency: columns on demand, then the ladder

**Part A - columns load on demand.** From cold, a one-column query, then residency by column: `gross_amount` resident, plus `event_key` because the cache-bypass filter runs through the `dim_event` relationship - relationship columns always load. Then the medium query: seven columns resident. Watch the temperature column: the two columns touched twice sit near 1.9, the five touched once near 1.0, the five never touched have none. Direct Lake loads per column, warms per access, and the columns nobody queries never cost anything.

**Part B - the four states.** One table, the light query, five measurements in sequence. This runs on `fragmented` (`LADDER_TABLE`), deliberately: on `base` the light query costs 0.55s cold and 0.38s warm, so there is nothing for the states to divide - in rehearsal the ladder on `base` was flat. On `fragmented` transcoding costs 12s, so the states separate.

| State | How we get there | Expect (rehearsal on base; fragmented to be calibrated) |
|---|---|---|
| cold | after clearValues | full transcoding of 4,000 row groups, about 12s |
| semiwarm | append one day, reframe | the 4,000 existing segments stay resident, only the 32 new row groups transcode: about 1s |
| warm | one more query | about 0.4s |
| hot | same again, cache bypassed | same as warm - VertiScan's cache only matters when the scan itself is expensive |
| result cache | same again, cache allowed | same again - a 0.4s query has nothing left to cache |

The readout that always works, on any table, is temperature: in rehearsal on `base` the average climbed 0.97, 1.87, 2.76 across the states, and the three queried columns ended at 4.47 after five accesses while nine columns never loaded at all. Residency and temperature are printed at each step.

**Stage time:** about 2 minutes.


In [21]:
cold_reset()
timed_query(BASE, "single", "one column (+ event_key via relationship)"); residency_by_column(BASE)
timed_query(BASE, "medium", "seven columns, five of them cold"); residency_by_column(BASE)


StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 28, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  one column (+ event_key via relationship) single     1.48s


SynapseWidget(Synapse.DataFrame, 100f76a8-04c2-4b77-954c-234cdb19206f)

  seven columns, five of them cold   medium     0.65s


SynapseWidget(Synapse.DataFrame, 20178f7c-e811-40c6-93d2-35ca449ad707)

In [22]:
LT = T(LADDER_TABLE)
ladder = []
def step(state, secs):
    r = residency(LT); ladder.append(dict(state=state, seconds=secs, **r))
cold_reset()
step("cold",     timed_query(LT, "light", "cold"))
snapshot(LT)
build_increment(LT).write.format("delta").mode("append").saveAsTable(LT); reframe()
r = residency(LT); print(f"  after append + reframe, before querying: {r['resident']}/{r['segments']} resident")
step("semiwarm", timed_query(LT, "light", "semiwarm (after append + reframe)"))
step("warm",     timed_query(LT, "light", "warm"))
results_table(ladder)
residency_by_column(LT)
restore(LT); reframe()

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 29, Finished, Available, Finished, False)

⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'clearValues' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  cold state induced (dimensions warm, facts cold)
  cold                               light     14.82s
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
  after append + reframe, before querying: 12000/48096 resident
  semiwarm (after append + reframe)  light      1.25s
  warm                               light      0.51s


SynapseWidget(Synapse.DataFrame, 88d916dd-0d5f-418d-9438-79578133a721)

SynapseWidget(Synapse.DataFrame, 0299f283-93ac-41be-a97e-86a95fe7bbba)

  fact_ticket_sales_fragmented restored to version 14
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.


---
## Reset - restore every fact to its pristine version

In [23]:
# Reset - restore every fact this session wrote to, then check every table against the Delta cache
for t in FACTS:
    if t not in PRISTINE:
        print(f"  {t}: not written in this session, nothing to restore")
        continue
    current = spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]
    if current == PRISTINE[t]:
        print(f"  {t}: already at version {current}")
        continue
    restore(t)
reframe()

bad = []
for v in VARIANTS:
    now = spark.sql(f"DESCRIBE DETAIL {T(v)}").collect()[0]["numFiles"]
    if now != delta_row(v)["files"]:
        bad.append(f"{v}: cache says {delta_row(v)['files']} files, table has {now}")
print("All tables match the cache" if not bad else "WARNING - check these tables:\n  " + "\n  ".join(bad))

StatementMeta(, 6a0082d4-dc99-4365-9606-5c96c81f4b9a, 30, Finished, Available, Finished, False)

  fact_ticket_sales_base restored to version 74
  fact_ticket_sales_many_files: not written in this session, nothing to restore
  fact_ticket_sales_small_rowgroups: not written in this session, nothing to restore
  fact_ticket_sales_fragmented restored to version 14
  fact_ticket_sales_novorder: not written in this session, nothing to restore
  fact_ticket_sales_part_month restored to version 26
  fact_ticket_sales_part_date restored to version 25
⌛ Refresh of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is in progress...
🟢 Refresh 'full' of the 'mosaic_direct_lake_demo' semantic model within the 'FabCon Barcelona' workspace is complete.
All tables match the cache
